# 06 - Multi-model consolidation

Reads from `ai_narratives_original` and the CSVs produced by notebook 05.
Writes a single Excel workbook with an Index sheet and one sheet per analysis.

## 1. Setup

In [1]:
import sys
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

REPO = Path.cwd().parent
sys.path.insert(0, str(REPO / 'src'))
from pain_narratives.analysis import revision_data_layer as dl

OUT = Path.cwd() / 'outputs'
OUT.mkdir(exist_ok=True, parents=True)
WORKBOOK = OUT / 'AINarratives_Multimodel_Consolidated.xlsx'

MODELS = dl.available_models()
print(f'Models: {MODELS}')


Models: ['claude-sonnet-4-5-thinking', 'deepseek-r1', 'gpt-5']


## 2. Sample characteristics (T1)

In [2]:
dem = dl.load_demographics_from_schema()
real = dl.load_real_from_schema()

def _counts(series):
    return series.value_counts(dropna=False).to_dict()

t1 = pd.DataFrame([
    {'Metric': 'N (patients)', 'Value': len(dem)},
    {'Metric': 'Age mean (SD)', 'Value': f"{dem['age'].mean():.1f} ({dem['age'].std():.1f})"},
    {'Metric': 'Age range', 'Value': f"{int(dem['age'].min())} - {int(dem['age'].max())}"},
    {'Metric': 'Word count mean (SD)', 'Value': f"{real['word_count'].mean():.1f} ({real['word_count'].std():.1f})"},
    {'Metric': 'Gender distribution', 'Value': '; '.join(f"{k}={v}" for k, v in _counts(dem['gender']).items())},
    {'Metric': 'Marital status distribution', 'Value': '; '.join(f"{k}={v}" for k, v in _counts(dem['marital_status']).items())},
    {'Metric': 'Employment distribution', 'Value': '; '.join(f"{k}={v}" for k, v in _counts(dem['employment_status']).items())},
    {'Metric': 'Education distribution', 'Value': '; '.join(f"{k}={v}" for k, v in _counts(dem['education_level']).items())},
])
print(t1.to_string(index=False))


                     Metric                                                                                                                                                                                                                                                                                                                                                                                Value
               N (patients)                                                                                                                                                                                                                                                                                                                                                                                  152
              Age mean (SD)                                                                                                                                                                           

## 2b. LLM dimension descriptives per model (T2)

Per-model mean / SD / min / max for the two LLM-generated dimension scores
(Severidad del dolor and Discapacidad). One row per (model, dimension), using
the per-narrative mean across runs as the unit. Matches the paper's T2 table
but extended to all three models.


In [3]:
rows = []
for m in MODELS:
    df_m = dl.load_synth_from_schema(m)
    for d_label, d_col in (('Severidad del dolor', 'severidad_score'),
                            ('Discapacidad', 'discapacidad_score')):
        per_pid = df_m.groupby('participant_id')[d_col].mean()
        rows.append({
            'dimension': d_label,
            'model': m,
            'n': per_pid.notna().sum(),
            'mean': per_pid.mean(),
            'sd': per_pid.std(),
            'min': per_pid.min(),
            'max': per_pid.max(),
            'n_runs': df_m['run_number'].nunique(),
        })
t2 = pd.DataFrame(rows)
print(t2.round(2).to_string(index=False))


          dimension                      model   n  mean   sd  min   max  n_runs
Severidad del dolor claude-sonnet-4-5-thinking 152  7.52 1.47 3.00 10.00       3
       Discapacidad claude-sonnet-4-5-thinking 151  7.25 1.79 1.00 10.00       3
Severidad del dolor                deepseek-r1 136  8.09 1.32 3.00 10.00       3
       Discapacidad                deepseek-r1 136  8.37 1.43 3.67 10.00       3
Severidad del dolor                      gpt-5 152  7.55 1.59 3.00 10.00       3
       Discapacidad                      gpt-5 152  7.03 1.88 2.00  9.67       3


## 3. Questionnaire descriptives (T3): real + synthetic per model

In [4]:
rows = []
for q_label, q_col in (('PCS total', 'pcs_total'),
                        ('BPI total mean', 'bpi_total_mean'),
                        ('TSK total', 'tsk_total')):
    rows.append({'questionnaire': q_label, 'source': 'real',
                 'n': len(real),
                 'mean': real[q_col].mean(),
                 'sd': real[q_col].std(),
                 'min': real[q_col].min(),
                 'max': real[q_col].max()})
    for m in MODELS:
        df_m = dl.load_synth_from_schema(m)
        per_pid = df_m.groupby('participant_id')[q_col].mean()
        rows.append({'questionnaire': q_label, 'source': m,
                     'n': len(per_pid),
                     'mean': per_pid.mean(),
                     'sd': per_pid.std(),
                     'min': per_pid.min(),
                     'max': per_pid.max()})
t3 = pd.DataFrame(rows)
print(t3.round(2).to_string(index=False))


 questionnaire                     source   n  mean    sd   min   max
     PCS total                       real 152 31.44 11.71  5.00 52.00
     PCS total claude-sonnet-4-5-thinking 152 35.67 13.59  3.00 51.67
     PCS total                deepseek-r1 152 38.08 11.43  5.67 51.00
     PCS total                      gpt-5 152 34.49 14.06  2.33 51.00
BPI total mean                       real 152  6.90  1.68  1.18  9.82
BPI total mean claude-sonnet-4-5-thinking 152  6.65  1.61  1.23  9.13
BPI total mean                deepseek-r1 152  7.45  1.52  2.17  9.57
BPI total mean                      gpt-5 152  6.53  1.79  1.17  9.07
     TSK total                       real 152 28.30  6.26 11.00 44.00
     TSK total claude-sonnet-4-5-thinking 152 32.58  7.33 12.67 43.00
     TSK total                deepseek-r1 152 36.47  6.86 18.00 44.00
     TSK total                      gpt-5 152 31.41  7.24 13.33 41.67


## 4. Real-vs-synthetic agreement (T3b) - read from notebook 05 output

In [5]:
t3b = pd.read_csv(OUT / '05_real_vs_synthetic_all.csv')
print(t3b.round(3).to_string(index=False))


                     model    questionnaire   n    mae   rmse  pearson_r  spearman
claude-sonnet-4-5-thinking        pcs_total 152 10.752 13.744      0.470     0.450
claude-sonnet-4-5-thinking   bpi_total_mean 152  0.967  1.241      0.726     0.734
claude-sonnet-4-5-thinking bpi_interference 152  1.207  1.639      0.611     0.666
claude-sonnet-4-5-thinking    bpi_intensity 152  1.210  1.494      0.710     0.664
claude-sonnet-4-5-thinking        tsk_total 152  7.114  8.785      0.368     0.321
               deepseek-r1        pcs_total 152 10.664 13.570      0.474     0.435
               deepseek-r1   bpi_total_mean 152  1.086  1.421      0.668     0.677
               deepseek-r1 bpi_interference 152  1.347  1.818      0.552     0.611
               deepseek-r1    bpi_intensity 152  1.191  1.559      0.650     0.643
               deepseek-r1        tsk_total 152  9.121 11.223      0.312     0.281
                     gpt-5        pcs_total 152 10.963 14.029      0.444     0.419
    

## 5. Cronbach alpha (T5) - read from notebook 05 output

In [6]:
t5 = pd.read_csv(OUT / '05_cronbach_alpha.csv')
print(t5.round(3).to_string(index=False))


        source                      model  pcs_alpha  bpi_interference_alpha  bpi_intensity_alpha  tsk_alpha  n_runs
          real                          -      0.928                   0.831                0.938      0.846       1
synthetic_mean claude-sonnet-4-5-thinking      0.986                   0.963                0.974      0.953       3
  synthetic_sd claude-sonnet-4-5-thinking      0.000                   0.001                0.001      0.001       3
synthetic_mean                deepseek-r1      0.972                   0.923                0.952      0.956       3
  synthetic_sd                deepseek-r1      0.001                   0.003                0.004      0.003       3
synthetic_mean                      gpt-5      0.980                   0.963                0.966      0.935       3
  synthetic_sd                      gpt-5      0.001                   0.001                0.004      0.001       3


## 6. LLM consistency across runs (T7)

In [7]:
t7 = pd.read_csv(OUT / '05_llm_consistency.csv')
print(t7.round(3).to_string(index=False))


                     model  questionnaire  n_runs  mean_sd_across_runs  min_sd  max_sd
claude-sonnet-4-5-thinking      pcs_total       3                2.039     0.0   8.505
claude-sonnet-4-5-thinking bpi_total_mean       3                0.214     0.0   0.794
claude-sonnet-4-5-thinking      tsk_total       3                1.644     0.0   6.658
               deepseek-r1      pcs_total       3                2.325     0.0   7.000
               deepseek-r1 bpi_total_mean       3                0.250     0.0   1.150
               deepseek-r1      tsk_total       3                1.767     0.0   5.859
                     gpt-5      pcs_total       3                2.223     0.0   6.658
                     gpt-5 bpi_total_mean       3                0.205     0.0   0.611
                     gpt-5      tsk_total       3                1.819     0.0   8.145


## 7. Synthetic dim vs real questionnaire (T11)

In [8]:
t11 = pd.read_csv(OUT / '05_dim_vs_real_quest.csv')
# Show a pivot: model x dim x quest -> rho
pivot = t11.pivot_table(index=['model','dimension'], columns='real_questionnaire',
                        values='spearman_rho').round(3)
print(pivot.to_string())


real_questionnaire                             bpi_intensity_mean  bpi_interference_mean  bpi_total_mean  pcs_total  tsk_total
model                      dimension                                                                                          
claude-sonnet-4-5-thinking discapacidad_score               0.512                  0.542           0.581      0.358      0.203
                           severidad_score                  0.520                  0.549           0.603      0.357      0.207
deepseek-r1                discapacidad_score               0.571                  0.527           0.600      0.412      0.291
                           severidad_score                  0.548                  0.493           0.565      0.391      0.173
gpt-5                      discapacidad_score               0.525                  0.553           0.608      0.311      0.199
                           severidad_score                  0.589                  0.551           0.631      0

## 8. Headline numbers

In [9]:
hn_rows = [
    {'Metric': 'Schema source', 'Value': 'ai_narratives_original'},
    {'Metric': 'Patients (N)', 'Value': len(real)},
    {'Metric': 'Models analyzed', 'Value': ', '.join(MODELS)},
    {'Metric': 'Runs per model', 'Value': '3'},
]
for m in MODELS:
    pcs_r = t3b.query("model == @m and questionnaire == 'pcs_total'")['pearson_r'].iloc[0]
    bpi_r = t3b.query("model == @m and questionnaire == 'bpi_total_mean'")['pearson_r'].iloc[0]
    tsk_r = t3b.query("model == @m and questionnaire == 'tsk_total'")['pearson_r'].iloc[0]
    hn_rows.append({'Metric': f'Pearson r vs real ({m}): PCS / BPI / TSK',
                    'Value': f'{pcs_r:.3f} / {bpi_r:.3f} / {tsk_r:.3f}'})
# Cronbach summary line per source
for _, row in t5.iterrows():
    hn_rows.append({'Metric': f"Cronbach alpha ({row['source']}, {row['model']})",
                    'Value': f"PCS={row['pcs_alpha']:.3f}  BPI_int={row['bpi_interference_alpha']:.3f}  "
                             f"BPI_inten={row['bpi_intensity_alpha']:.3f}  TSK={row['tsk_alpha']:.3f}"})
headline = pd.DataFrame(hn_rows)
print(headline.to_string(index=False))


                                                         Metric                                                Value
                                                  Schema source                               ai_narratives_original
                                                   Patients (N)                                                  152
                                                Models analyzed       claude-sonnet-4-5-thinking, deepseek-r1, gpt-5
                                                 Runs per model                                                    3
Pearson r vs real (claude-sonnet-4-5-thinking): PCS / BPI / TSK                                0.470 / 0.726 / 0.368
               Pearson r vs real (deepseek-r1): PCS / BPI / TSK                                0.474 / 0.668 / 0.312
                     Pearson r vs real (gpt-5): PCS / BPI / TSK                                0.444 / 0.699 / 0.344
                                       Cronbach alpha (real, -) 

## 9. Write the workbook

In [10]:
INDEX_SHEETS = [
    ('Index', 'This sheet. Map of every analysis sheet in this workbook.'),
    ('Headline_numbers', 'Top-line summary across all models and analyses.'),
    ('T1_Sample_Characteristics', 'Patient demographics (N=152).'),
    ('T2_Dimension_Descriptives', 'Per-model mean / SD / range for LLM dimension scores (Severidad, Discapacidad).'),
    ('T3_Questionnaire_Descriptives', 'Real + per-model synthetic means / SDs for PCS / BPI / TSK.'),
    ('T3b_Real_vs_Synth_Agreement', 'Per-model MAE / RMSE / Pearson r / Spearman rho.'),
    ('T5_Cronbach_Alpha', 'Real (single) + per-model synthetic (mean+/-SD across runs) alpha.'),
    ('T7_LLM_Consistency', 'Within-model SD across runs per participant.'),
    ('T11_Dim_vs_Real_Quest', 'LLM dimension scores correlated with real patient questionnaires.'),
]
index_df = pd.DataFrame(INDEX_SHEETS, columns=['Sheet', 'Description'])

with pd.ExcelWriter(WORKBOOK, engine='openpyxl') as writer:
    index_df.to_excel(writer, sheet_name='Index', index=False)
    headline.to_excel(writer, sheet_name='Headline_numbers', index=False)
    t1.to_excel(writer, sheet_name='T1_Sample_Characteristics', index=False)
    t2.to_excel(writer, sheet_name='T2_Dimension_Descriptives', index=False)
    t3.to_excel(writer, sheet_name='T3_Questionnaire_Descriptives', index=False)
    t3b.to_excel(writer, sheet_name='T3b_Real_vs_Synth_Agreement', index=False)
    t5.to_excel(writer, sheet_name='T5_Cronbach_Alpha', index=False)
    t7.to_excel(writer, sheet_name='T7_LLM_Consistency', index=False)
    t11.to_excel(writer, sheet_name='T11_Dim_vs_Real_Quest', index=False)

print(f'Wrote {WORKBOOK}')
print(f'Generated at {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')


Wrote /Users/gferreira/Documents/my_repos/pain-narratives-app-public/notebooks/outputs/AINarratives_Multimodel_Consolidated.xlsx
Generated at 2026-05-26 15:43:39
